# 15장. 코사인 유사도

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 3장, 4장, 13장 |
| 이 장의 결과물 | `cosine_similarity(doc_matrix, query_vec)` 함수, 과제 예시 형식의 계산 과정 로그 |
| 관련 학습 목표 | ③ 코사인 유사도가 벡터의 내적과 정규화로 계산되는 원리 |

**학습 목표**
- 문서 검색에서 거리 대신 **각도**를 쓰는 이유를 실제 데이터로 보여 줄 수 있다.
- 코사인 유사도 공식을 내적 → 노름 → 나눗셈 순서로 직접 계산하고 과정을 출력할 수 있다.
- L2 정규화된 벡터에서는 내적이 곧 코사인 유사도인 이유를 설명할 수 있다.
- 모든 문서의 유사도를 `X @ q` 한 번으로 계산하고, sklearn 결과와 비교해 검증할 수 있다.
- 쿼리 벡터가 0일 때 `nan` 없이 처리할 수 있다.

3장에서 순수 파이썬으로 배운 코사인 유사도를, 이번에는 **실제 TF-IDF 행렬**(문서 수천 개 × 단어 수천 개)에서 NumPy로 계산합니다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

# ── 8~10장에서 만든 전처리 파이프라인 (src/preprocess.py로 옮길 내용) ──
from konlpy.tag import Okt
from nltk.corpus import stopwords

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}
EN_STOP = frozenset(stopwords.words("english"))
DOMAIN_STOPWORDS_EN = frozenset({
    "would", "could", "might", "may", "also", "like", "get", "got", "go", "going", "know", "think",
    "see", "say", "make", "want", "look", "one", "well", "much", "many", "even", "way", "something",
    "anyone", "since", "still", "really", "right", "another", "cant",
})
STOPWORDS_KO = frozenset({
    "하다", "되다", "있다", "없다", "이다", "않다",
    "등", "것", "수", "중", "및", "때", "후", "내", "듯", "제", "개", "의", "로", "종합",
})
_okt = Okt()


def clean_en(text: str) -> str:
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


class EnglishPreprocessor:
    def __init__(self, use_domain_stopwords: bool = True, min_len: int = 2, max_len: int = 20):
        self.stopwords = EN_STOP | (DOMAIN_STOPWORDS_EN if use_domain_stopwords else frozenset())
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for tok in clean_en(text).split():
            if tok in self.stopwords:
                continue
            tok = tok.removesuffix("'s").replace("'", "")
            if self.min_len <= len(tok) <= self.max_len and tok not in self.stopwords:
                tokens.append(tok)
        return tokens


class KoreanPreprocessor:
    def __init__(self, tokenizer: str = "okt", use_stopwords: bool = True):
        if tokenizer not in ("okt", "whitespace"):
            raise ValueError(f"tokenizer는 'okt' 또는 'whitespace': {tokenizer}")
        self.tokenizer = tokenizer
        self.stopwords = STOPWORDS_KO if use_stopwords else frozenset()

    def tokenize(self, text: str) -> list[str]:
        if self.tokenizer == "whitespace":
            return clean_ko(text).lower().split()
        return [w.lower() for w, p in _okt.pos(clean_ko(text), norm=True, stem=True)
                if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]

    def __call__(self, text: str) -> list[str]:
        return [t for t in self.tokenize(text) if t not in self.stopwords and not t.isdigit()]

# ── 로드 + 전처리를 한 번에 (src/corpus.py로 옮길 내용) ──
MIN_TOKENS = {"en": 5, "ko": 2}   # 전처리 후 토큰이 이보다 적은 문서는 뺀다 (짧은 답글, 주제어 없는 제목)
MIN_DF = {"en": 3, "ko": 2}       # 어휘에 넣을 최소 문서 빈도 (11장)


def load_corpus(lang: str, tokenizer: str = "okt") -> pd.DataFrame:
    """열: text, label, tokens. lang은 "en" 또는 "ko"."""
    df = load_english() if lang == "en" else load_korean()
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor(tokenizer)
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df

# ── 13장에서 만든 NumPy TF-IDF (src/tfidf.py로 옮길 내용) ──
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)

import time

en = load_corpus("en")
vec_raw = NumpyTfidf(min_df=MIN_DF["en"], norm=None)    # 정규화하지 않은 TF×IDF (길이 정보가 남아 있음)
X_raw = vec_raw.fit_transform(en["tokens"].tolist())
vec = NumpyTfidf(min_df=MIN_DF["en"])                   # L2 정규화한 TF-IDF (검색에 쓰는 것)
X = vec.fit_transform(en["tokens"].tolist())
prep_en = en.attrs["preprocessor"]
print("문서-단어 행렬:", X.shape)

문서-단어 행렬: (4646, 13387)


같은 어휘와 IDF로 두 가지 행렬을 만들었습니다.
- `X_raw`: TF × IDF까지만 계산하고 **정규화하지 않은** 행렬. 긴 문서일수록 벡터가 깁니다.
- `X`: 각 행을 L2 노름으로 나눈 행렬. 모든 문서 벡터의 길이가 1입니다.

## 15.1 거리 대신 각도를 쓰는 이유

### 문서 길이는 제각각이다

In [2]:
lengths = en["tokens"].map(len)
raw_norms = np.linalg.norm(X_raw, axis=1)
print("문서당 토큰 수:", lengths.describe()[["min", "25%", "50%", "75%", "max"]].astype(int).to_dict())
print(f"토큰 수와 TF×IDF 벡터 길이의 상관계수: {np.corrcoef(lengths, raw_norms)[0, 1]:.3f}")
print("정규화한 X의 벡터 길이:", np.unique(np.linalg.norm(X, axis=1).round(6)))

문서당 토큰 수: {'min': 5, '25%': 20, '50%': 38, '75%': 75, 'max': 6417}
토큰 수와 TF×IDF 벡터 길이의 상관계수: 0.914
정규화한 X의 벡터 길이: [1.]


문서가 길수록 단어 횟수가 커지고, 그래서 TF×IDF 벡터도 길어집니다. 3장에서 본 것처럼 **유클리드 거리는 이 길이 차이에 민감**합니다. 실제 검색에서 어떤 일이 생기는지 비교합니다. 같은 쿼리로 (1) 유클리드 거리가 가장 가까운 문서와 (2) 코사인 유사도가 가장 큰 문서를 뽑아 봅니다.

In [3]:
query = "space shuttle launch"
q_tokens = prep_en(query)
q_raw = vec_raw.transform([q_tokens])[0]

dist = np.linalg.norm(X_raw - q_raw, axis=1)                         # 유클리드 거리 (작을수록 가깝다)
cos = (X_raw @ q_raw) / (np.linalg.norm(X_raw, axis=1) * np.linalg.norm(q_raw))   # 코사인 (클수록 비슷)


def show(order, title):
    print(title)
    for i in order:
        has = [t for t in q_tokens if t in en.loc[i, "tokens"]]
        print(f"  doc {i:>4} [{en.loc[i, 'label']:<18}] 토큰 {lengths[i]:>4}개  거리 {dist[i]:6.2f}  코사인 {cos[i]:.3f}  쿼리 단어 {has}")


print(f"Query: {query!r} → 토큰 {q_tokens}\n")
show(np.argsort(dist, kind="stable")[:5], "▶ 유클리드 거리가 가장 가까운 5개")
show(np.argsort(-cos, kind="stable")[:5], "\n▶ 코사인 유사도가 가장 큰 5개")

Query: 'space shuttle launch' → 토큰 ['space', 'shuttle', 'launch']

▶ 유클리드 거리가 가장 가까운 5개
  doc 4008 [sci.space         ] 토큰    6개  거리  10.27  코사인 0.000  쿼리 단어 []
  doc 1854 [sci.space         ] 토큰    5개  거리  10.81  코사인 0.000  쿼리 단어 []
  doc 1502 [comp.windows.x    ] 토큰    5개  거리  10.98  코사인 0.000  쿼리 단어 []
  doc 1564 [rec.sport.hockey  ] 토큰    6개  거리  11.02  코사인 0.000  쿼리 단어 []
  doc 2605 [rec.sport.baseball] 토큰    6개  거리  11.50  코사인 0.000  쿼리 단어 []

▶ 코사인 유사도가 가장 큰 5개
  doc 3088 [sci.space         ] 토큰   24개  거리  25.32  코사인 0.423  쿼리 단어 ['shuttle', 'launch']
  doc 1830 [sci.space         ] 토큰 3392개  거리 972.32  코사인 0.418  쿼리 단어 ['space', 'shuttle', 'launch']
  doc 4306 [sci.space         ] 토큰   59개  거리  47.66  코사인 0.406  쿼리 단어 ['shuttle', 'launch']
  doc 1292 [sci.space         ] 토큰   57개  거리  37.98  코사인 0.406  쿼리 단어 ['space', 'shuttle', 'launch']
  doc   38 [sci.space         ] 토큰   32개  거리  27.68  코사인 0.399  쿼리 단어 ['space', 'shuttle', 'launch']


유클리드 거리로 고른 문서들은 **아주 짧고, 쿼리 단어가 거의 없는 문서**입니다. 짧은 문서는 벡터가 원점 근처에 있고, 쿼리 벡터도 단어 3개뿐이라 원점 근처에 있으므로 "가깝다"고 나옵니다. 내용이 비슷해서가 아니라 **둘 다 짧아서** 가까운 것입니다.

코사인 유사도는 벡터의 **방향**만 봅니다. 쿼리 단어를 실제로 담고 있는 우주 게시판 글이 위로 올라오고, 문서 길이는 짧은 글부터 3,000토큰이 넘는 긴 글까지 제각각입니다. 길이가 아니라 **쿼리 단어가 문서에서 차지하는 비중**으로 순위가 정해지기 때문입니다. 문서 검색에서 코사인을 쓰는 이유가 바로 이것입니다.

In [4]:
order = np.argsort(dist, kind="stable")
print(f"거리 기준 상위 50개 문서의 평균 토큰 수: {lengths[order[:50]].mean():.1f}")
order = np.argsort(-cos, kind="stable")
print(f"코사인 기준 상위 50개 문서의 평균 토큰 수: {lengths[order[:50]].mean():.1f}")
print(f"전체 문서 평균 토큰 수                 : {lengths.mean():.1f}")

거리 기준 상위 50개 문서의 평균 토큰 수: 5.7
코사인 기준 상위 50개 문서의 평균 토큰 수: 338.7
전체 문서 평균 토큰 수                 : 98.8


## 15.2 공식을 단계별로 계산하기

$$\cos(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\lVert \mathbf{q} \rVert \, \lVert \mathbf{d} \rVert} = \frac{\sum_t q_t d_t}{\sqrt{\sum_t q_t^2} \, \sqrt{\sum_t d_t^2}}$$

### 먼저 미니 코퍼스로
값이 작아서 손으로 확인할 수 있는 미니 코퍼스부터 계산합니다. 문서 5개로 TF-IDF(정규화 전)를 만들고, 쿼리 `"space rocket"`과 doc 0의 유사도를 단계별로 구합니다.

In [5]:
# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

mini_tokens = [EnglishPreprocessor()(d) for d in MINI_DOCS]
mini_vec = NumpyTfidf(norm=None).fit(mini_tokens)
M = mini_vec.transform(mini_tokens)
mq = mini_vec.transform([EnglishPreprocessor()("space rocket")])[0]
terms = mini_vec.terms_

print("어휘:", list(terms))
print("쿼리 벡터 :", mq.round(3))
print("doc 0 벡터:", M[0].round(3))

products = mq * M[0]                                  # ① 같은 위치끼리 곱하기
dot = products.sum()                                  # ② 모두 더하기 = 내적
q_norm = np.sqrt((mq ** 2).sum())                     # ③ 쿼리 길이
d_norm = np.sqrt((M[0] ** 2).sum())                   # ④ 문서 길이
print("\n① q_t × d_t (0이 아닌 것):", {t: round(float(p), 3) for t, p in zip(terms, products) if p})
print(f"② 내적      = {dot:.3f}")
print(f"③ ||Query|| = {q_norm:.3f}")
print(f"④ ||문서||  = {d_norm:.3f}")
print(f"⑤ 코사인    = {dot:.3f} / ({q_norm:.3f} × {d_norm:.3f}) = {dot / (q_norm * d_norm):.3f}")

어휘: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'new', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched']
쿼리 벡터 : [0.    0.    0.    0.    0.    0.    0.    0.    0.    2.099 1.405 0.
 0.    0.    0.   ]
doc 0 벡터: [0.    0.    0.    0.    0.    1.693 0.    0.    0.    2.099 1.405 0.
 0.    0.    0.   ]

① q_t × d_t (0이 아닌 것): {'rocket': 4.404, 'space': 1.975}
② 내적      = 6.380
③ ||Query|| = 2.526
④ ||문서||  = 3.041
⑤ 코사인    = 6.380 / (2.526 × 3.041) = 0.831


doc 0은 `launched`, `rocket`, `space`를 가졌고 쿼리는 `rocket`, `space`를 가졌습니다. 내적에는 **둘 다 가진 단어(rocket, space)만** 기여합니다. `launched`는 문서 쪽 길이(④)를 늘려서 유사도를 1보다 작게 만듭니다.

### 실제 데이터로: 과제 결과 예시 형식
과제의 "코사인 유사도 계산 과정" 예시와 같은 모양으로 출력하는 함수를 만듭니다. 벡터 차원이 1만 개가 넘으므로 **0이 아닌 항목만** 보여 줍니다.

In [6]:
def nonzero_items(v: np.ndarray, terms, top: int = 6) -> str:
    idx = np.flatnonzero(v)
    idx = idx[np.argsort(-v[idx], kind="stable")][:top]
    more = "" if len(np.flatnonzero(v)) <= top else f", ... (0이 아닌 값 총 {np.count_nonzero(v)}개)"
    return "{" + ", ".join(f"{terms[j]}: {v[j]:.2f}" for j in idx) + more + "}"


def explain_cosine(query: str, doc_index: int) -> None:
    q = vec_raw.transform([prep_en(query)])[0]
    d = X_raw[doc_index]
    common = np.flatnonzero((q != 0) & (d != 0))
    dot, qn, dn = q @ d, np.linalg.norm(q), np.linalg.norm(d)
    print("=== 코사인 유사도 계산 예시 ===")
    print(f'Query: "{query}"  →  토큰 {prep_en(query)}')
    print(f"대상: doc {doc_index} [{en.loc[doc_index, 'label']}]\n")
    print("Query 벡터 (0이 아닌 값):", nonzero_items(q, vec_raw.terms_))
    print("문서 벡터  (큰 값 6개)  :", nonzero_items(d, vec_raw.terms_))
    print("공통 단어의 곱 q_t×d_t  :", {vec_raw.terms_[j]: round(float(q[j] * d[j]), 2) for j in common})
    print("\n코사인 유사도 = (Query · 문서) / (||Query|| × ||문서||)")
    print(f"            = {dot:.3f} / ({qn:.3f} × {dn:.3f}) = {dot / (qn * dn):.3f}")


best = int(np.argmax(cos))
explain_cosine("space shuttle launch", best)

=== 코사인 유사도 계산 예시 ===
Query: "space shuttle launch"  →  토큰 ['space', 'shuttle', 'launch']
대상: doc 3088 [sci.space]

Query 벡터 (0이 아닌 값): {launch: 4.66, shuttle: 4.64, space: 3.42}
문서 벡터  (큰 값 6개)  : {launch: 9.31, shuttle: 9.28, campaign: 7.65, elv: 7.50, perot: 7.50, nixon: 6.88, ... (0이 아닌 값 총 21개)}
공통 단어의 곱 q_t×d_t  : {'launch': 43.37, 'shuttle': 43.06}

코사인 유사도 = (Query · 문서) / (||Query|| × ||문서||)
            = 86.424 / (7.410 × 27.550) = 0.423


- 벡터 차원은 어휘 수(1만 개 이상)지만, 쿼리에서 0이 아닌 값은 쿼리 단어 수만큼뿐입니다.
- 그래서 내적은 **쿼리와 문서에 함께 있는 단어**의 곱만 더한 값입니다.
- 문서 쪽 노름에는 쿼리와 상관없는 단어까지 모두 들어갑니다. 쿼리 단어가 문서에서 차지하는 **비중**이 클수록 유사도가 커집니다.

## 15.3 정규화된 벡터에서는 내적 = 코사인

미리 L2 정규화해 두면 $\lVert \hat{\mathbf q} \rVert = \lVert \hat{\mathbf d} \rVert = 1$이므로

$$\cos(\mathbf q, \mathbf d) = \frac{\mathbf q \cdot \mathbf d}{\lVert \mathbf q \rVert \lVert \mathbf d \rVert} = \frac{\mathbf q}{\lVert \mathbf q \rVert} \cdot \frac{\mathbf d}{\lVert \mathbf d \rVert} = \hat{\mathbf q} \cdot \hat{\mathbf d}$$

정규화는 벡터의 **길이만 1로 바꾸고 방향은 그대로** 두기 때문에, 각도(코사인)도 그대로입니다.

In [7]:
q_hat = vec.transform([q_tokens])[0]                  # 정규화된 쿼리 벡터
dot_normalized = X @ q_hat                            # 정규화된 벡터끼리의 내적

print(f"정규화 전 벡터로 계산한 코사인 (doc {best}): {cos[best]:.6f}")
print(f"정규화된 벡터의 내적         (doc {best}): {dot_normalized[best]:.6f}")
print("모든 문서에서 같은가 (오차 1e-12 이내):", np.allclose(cos, dot_normalized, rtol=0, atol=1e-12))

정규화 전 벡터로 계산한 코사인 (doc 3088): 0.423335
정규화된 벡터의 내적         (doc 3088): 0.423335
모든 문서에서 같은가 (오차 1e-12 이내): True


그래서 검색 시스템은 문서 행렬을 **한 번만 정규화해서 저장**해 두고, 쿼리가 들어올 때마다 쿼리 벡터 하나만 정규화한 뒤 내적만 계산하면 됩니다.

## 15.4 모든 문서와 한 번에 계산하기: `X @ q`

`(n, V)` 행렬과 `(V,)` 벡터를 곱하면 각 행과의 내적이 모인 `(n,)` 벡터가 나옵니다(3장, 4장). 루프와 속도를 비교합니다.

In [8]:
t0 = time.perf_counter()
loop_scores = np.array([X[i] @ q_hat for i in range(X.shape[0])])     # 문서마다 한 번씩
t1 = time.perf_counter()
batch_scores = X @ q_hat                                               # 행렬 × 벡터 한 번
t2 = time.perf_counter()

print(f"X.shape={X.shape}, q.shape={q_hat.shape} → 결과 shape {batch_scores.shape}")
print(f"문서별 루프 : {t1 - t0:.4f}초")
print(f"X @ q       : {t2 - t1:.4f}초  (약 {(t1 - t0) / (t2 - t1):.0f}배 빠름, 실행 환경마다 다름)")
print("결과 같음?  :", np.allclose(loop_scores, batch_scores))

X.shape=(4646, 13387), q.shape=(13387,) → 결과 shape (4646,)
문서별 루프 : 0.0185초
X @ q       : 0.0106초  (약 2배 빠름, 실행 환경마다 다름)
결과 같음?  : True


### 정규화 여부와 상관없이 쓸 수 있는 함수
문서 행렬이 정규화되어 있지 않아도 동작하도록, 노름으로 나누는 부분까지 넣은 함수를 만듭니다. 이 함수가 `src/search.py`의 `cosine_similarity`가 됩니다. 0으로 나누는 경우는 15.6절에서 다룹니다.

In [9]:
def cosine_similarity(doc_matrix: np.ndarray, query_vec: np.ndarray) -> np.ndarray:
    """문서 행렬 (n, V)와 쿼리 벡터 (V,)의 코사인 유사도 (n,). 0 벡터와의 유사도는 0."""
    dots = doc_matrix @ query_vec                         # (n,) 내적
    doc_norms = np.linalg.norm(doc_matrix, axis=1)        # (n,) 문서 길이
    q_norm = np.linalg.norm(query_vec)                    # 스칼라
    denom = doc_norms * q_norm
    return np.divide(dots, denom, out=np.zeros_like(dots), where=denom > 0)


s_raw = cosine_similarity(X_raw, q_raw)
s_norm = cosine_similarity(X, q_hat)
print("정규화 전 행렬로 :", s_raw[best].round(6))
print("정규화 후 행렬로 :", s_norm[best].round(6))
print("전체 일치        :", np.allclose(s_raw, s_norm))

정규화 전 행렬로 : 0.423335
정규화 후 행렬로 : 0.423335
전체 일치        : True


`np.divide(a, b, out=..., where=조건)`은 조건이 True인 위치에서만 나누고, 나머지는 `out`에 미리 넣어 둔 값(여기서는 0)을 그대로 둡니다. 4장의 `np.where(norms == 0, 1, norms)`와 같은 목적의 다른 방법입니다.

## 15.5 sklearn으로 교차 검증하기

내 구현이 맞는지 sklearn의 `cosine_similarity`와 비교합니다. 과제 제약상 sklearn은 **검증 용도로만** 쓰고, 검색 시스템 자체에는 넣지 않습니다.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

queries = ["space shuttle launch", "hubble telescope mirror", "window server font", "pitcher fastball inning"]
for qtext in queries:
    q = vec_raw.transform([prep_en(qtext)])[0]
    ours = cosine_similarity(X_raw, q)
    theirs = sk_cosine(X_raw, q[None, :]).ravel()         # sklearn은 2차원 입력을 받는다
    print(f"{qtext!r:<28} 최대 오차 {np.abs(ours - theirs).max():.2e}")

'space shuttle launch'       최대 오차 2.22e-16


'hubble telescope mirror'    최대 오차 1.11e-16


'window server font'         최대 오차 1.11e-16


'pitcher fastball inning'    최대 오차 5.55e-17


오차가 부동소수점 반올림 수준이면 구현이 맞습니다.

## 15.6 쿼리 벡터가 0일 때

쿼리를 전처리했더니 **남는 단어가 없거나**(모두 불용어), **어휘 사전에 없는 단어만** 있으면 쿼리 벡터가 전부 0이 됩니다. 이때 공식 그대로 나누면 $0 / 0$이 되어 `nan`이 나옵니다.

In [11]:
for qtext in ["the and of", "zyzzyva qwertyuiop", "space zyzzyva"]:
    toks = prep_en(qtext)
    known = [t for t in toks if t in vec.vocabulary_]
    q = vec_raw.transform([toks])[0]
    with np.errstate(invalid="ignore", divide="ignore"):
        naive = (X_raw @ q) / (np.linalg.norm(X_raw, axis=1) * np.linalg.norm(q))
    safe = cosine_similarity(X_raw, q)
    print(f"{qtext!r:<22} 토큰 {toks} → 어휘에 있는 단어 {known}")
    print(f"{'':<22} 공식 그대로: nan {np.isnan(naive).sum():>4}개 | cosine_similarity: nan {np.isnan(safe).sum()}개, 최대 점수 {safe.max():.3f}")

'the and of'           토큰 [] → 어휘에 있는 단어 []
                       공식 그대로: nan 4646개 | cosine_similarity: nan 0개, 최대 점수 0.000


'zyzzyva qwertyuiop'   토큰 ['zyzzyva', 'qwertyuiop'] → 어휘에 있는 단어 []
                       공식 그대로: nan 4646개 | cosine_similarity: nan 0개, 최대 점수 0.000


'space zyzzyva'        토큰 ['space', 'zyzzyva'] → 어휘에 있는 단어 ['space']
                       공식 그대로: nan    0개 | cosine_similarity: nan 0개, 최대 점수 0.585


- `"the and of"`: 모두 불용어라 토큰이 0개 → 모든 점수 0. 검색 결과는 **빈 목록**을 돌려줘야 합니다(16장).
- `"zyzzyva qwertyuiop"`: 토큰은 있지만 어휘에 없는 단어(OOV)뿐 → 역시 0 벡터.
- `"space zyzzyva"`: 어휘에 있는 `space`만 반영되고 OOV는 조용히 무시됩니다. 사용자는 단어 하나가 무시된 줄 모를 수 있으므로, 16장에서는 "어휘에 없어 버린 단어"를 함께 보여 줍니다.

## 📝 리포트용 정리: 코사인 유사도 구현 (학습 목표 ③)

$$\cos(\mathbf q,\mathbf d) = \frac{\mathbf q\cdot\mathbf d}{\lVert\mathbf q\rVert\,\lVert\mathbf d\rVert}$$

| 단계 | NumPy 구현 | 설명 |
|---|---|---|
| 내적 | `doc_matrix @ query_vec` | 모든 문서와의 내적을 행렬-벡터 곱 한 번으로. 공통 단어만 기여 |
| 노름 | `np.linalg.norm(doc_matrix, axis=1)`, `np.linalg.norm(query_vec)` | 문서마다, 쿼리 하나 |
| 나눗셈 | `np.divide(..., where=denom > 0)` | 0 벡터(불용어·OOV만 있는 쿼리, 빈 문서)는 점수 0 |
| 최적화 | 문서 행렬을 미리 L2 정규화 | 정규화된 벡터끼리는 내적 = 코사인 |
| 검증 | `sklearn.metrics.pairwise.cosine_similarity`와 비교 | 오차가 부동소수점 반올림 수준 (15.5절) |
| 거리 대신 각도 | 유클리드 거리는 짧은 문서를 고른다 | 15.1절 비교 |

## ✍️ 실습 1: 코사인 점수 함수

`np.divide`를 쓰지 말고, `np.where`로 0 나눗셈을 막는 `cosine_scores(X, q)`를 작성하세요. 아래 테스트를 통과해야 합니다.

In [12]:
X_test = np.array([[3.0, 4.0, 0.0],
                   [0.0, 0.0, 0.0],      # 빈 문서
                   [1.0, 0.0, 0.0]])
q_test = np.array([3.0, 4.0, 0.0])


def check(func):
    got = func(X_test, q_test)
    ok1 = np.allclose(got, [1.0, 0.0, 0.6])
    ok2 = np.allclose(func(X_test, np.zeros(3)), 0.0) and not np.isnan(func(X_test, np.zeros(3))).any()
    print("기본 계산:", "✅" if ok1 else f"❌ {got}")
    print("0 벡터   :", "✅" if ok2 else "❌")

In [13]:
# 여기에 작성하세요

**정답**

In [14]:
def cosine_scores(X: np.ndarray, q: np.ndarray) -> np.ndarray:
    denom = np.linalg.norm(X, axis=1) * np.linalg.norm(q)
    return (X @ q) / np.where(denom > 0, denom, 1.0)     # 분모가 0이면 분자(내적)도 0이므로 결과는 0


check(cosine_scores)

기본 계산: ✅
0 벡터   : ✅


분모가 0인 경우는 문서나 쿼리가 0 벡터일 때뿐이고, 그때는 내적도 0입니다. 그래서 분모를 1로 바꿔 나누면 결과가 0이 됩니다.

## ✍️ 실습 2: 계산 과정 직접 출력하기

쿼리 `"hubble telescope mirror"`에 대해
1. 정규화된 행렬 `X`로 코사인 유사도 1위 문서를 찾고,
2. 그 문서와 쿼리의 **공통 단어**마다 `q_t`, `d_t`, `q_t × d_t`를 표로 출력한 뒤,
3. 곱의 합이 1위 점수와 같은지 확인하세요.

In [15]:
# 여기에 작성하세요

**정답**

In [16]:
qtext = "hubble telescope mirror"
qv = vec.transform([prep_en(qtext)])[0]
scores = X @ qv
top = int(np.argmax(scores))
common = np.flatnonzero((qv != 0) & (X[top] != 0))

print(f"1위: doc {top} [{en.loc[top, 'label']}] 점수 {scores[top]:.4f}")
print(f"{'단어':<12}{'q_t':>8}{'d_t':>8}{'q_t×d_t':>10}")
for j in common:
    print(f"{vec.terms_[j]:<12}{qv[j]:>8.4f}{X[top, j]:>8.4f}{qv[j] * X[top, j]:>10.4f}")
print(f"곱의 합 = {(qv[common] * X[top, common]).sum():.4f}")

1위: doc 1321 [sci.space] 점수 0.3920
단어               q_t     d_t   q_t×d_t
hubble        0.6114  0.2812    0.1719
mirror        0.5749  0.2645    0.1520
telescope     0.5437  0.1251    0.0680
곱의 합 = 0.3920


정규화된 벡터이므로 공통 단어의 곱을 더한 값이 곧 코사인 유사도입니다. 공통 단어가 아닌 차원은 어느 한쪽이 0이라 더할 필요가 없습니다.

## 🧐 퀴즈

**Q1.** 유클리드 거리로 검색하면 짧은 문서가 위로 올라오는 이유는?
<details><summary>▶ 정답 보기</summary>

짧은 문서는 단어 횟수가 작아 TF×IDF 벡터가 원점 가까이에 있고, 쿼리 벡터도 단어가 몇 개뿐이라 원점 가까이에 있습니다. 그래서 내용과 상관없이 "둘 다 짧다"는 이유로 거리가 가까워집니다. 코사인은 길이를 나눠 없애고 방향만 비교하므로 이 문제가 없습니다.
</details>

**Q2.** 쿼리 `"space shuttle"`과 문서의 코사인 유사도를 계산할 때, 문서에만 있는 단어 `orbit`은 분자와 분모 중 어디에 영향을 주나요?
<details><summary>▶ 정답 보기</summary>

분모(문서 노름)에만 영향을 줍니다. 쿼리 벡터의 `orbit` 값이 0이라 내적(분자)에는 기여하지 않고, 문서 노름을 키워서 유사도를 낮춥니다.
</details>

**Q3.** 문서 행렬을 미리 L2 정규화해 두면 검색할 때 무엇을 생략할 수 있나요?
<details><summary>▶ 정답 보기</summary>

문서 노름을 매번 계산하고 나누는 과정을 생략할 수 있습니다. 쿼리 벡터만 정규화하면 `X @ q`가 곧 코사인 유사도입니다.
</details>

**Q4.** 쿼리의 모든 단어가 불용어라서 토큰이 하나도 남지 않으면 어떻게 처리해야 하나요?
<details><summary>▶ 정답 보기</summary>

쿼리 벡터가 0 벡터이므로 공식 그대로 나누면 `0/0 = nan`이 됩니다. 분모가 0인 곳은 점수를 0으로 두고, 검색 결과로는 빈 목록을 돌려주며 사용자에게 이유를 알려 줍니다.
</details>

**Q5.** sklearn의 `cosine_similarity`를 검색 시스템 안에서 바로 쓰면 안 되는 이유는?
<details><summary>▶ 정답 보기</summary>

과제 제약이 "코사인 유사도는 NumPy 벡터 연산으로 직접 구현"이기 때문입니다. sklearn은 내 구현이 맞는지 비교하는 검증 용도로만 씁니다.
</details>

## ✅ 체크포인트
- [ ] 유클리드 거리와 코사인 유사도의 검색 결과 차이를 실데이터로 설명할 수 있다.
- [ ] 코사인 유사도를 내적 → 노름 → 나눗셈 순서로 계산하고 과정을 출력할 수 있다.
- [ ] "정규화된 벡터의 내적 = 코사인"을 식과 코드로 보여 줄 수 있다.
- [ ] `X @ q`로 모든 문서의 점수를 한 번에 계산할 수 있다.
- [ ] 0 벡터 쿼리를 `nan` 없이 처리할 수 있다.

## 🔗 과제 연결과 다음 장
- `cosine_similarity` 함수는 `src/search.py`에 들어가고, `explain_cosine`의 출력 형식은 과제 결과 예시 "코사인 유사도 계산 과정"과 README의 계산 예시에 씁니다.
- **16장. 검색기 완성**에서 이 함수에 쿼리 전처리, 상위 k개, 스니펫, CLI를 붙여 과제의 검색 시스템을 완성합니다.